In [3]:
# build_faiss_from_markdown.py
from pathlib import Path
from typing import List

# Loaders / splitters / vectors
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS 


from langchain_openai import OpenAIEmbeddings
# from langchain_nomic.embeddings import NomicEmbeddings  # optional

In [14]:
# --------- CONFIG ---------
MARKDOWN_INPUT = INPUT_DIR = Path("../data/ocr_md")    # .md file OR directory
EMBEDDINGS_DIRECTORY = "../vstore"               # output folder for FAISS
USE_OPENAI = True                               # True=OpenAI, False=Nomic local

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150
# --------------------------

In [15]:
def load_markdown_paths(root: Path) -> List[Path]:
    if root.is_file() and root.suffix.lower() == ".md":
        return [root]
    if root.is_dir():
        return sorted(root.rglob("*.md"))
    raise FileNotFoundError(f"Not a .md file or directory: {root}")

def load_docs(paths: List[Path]):
    docs = []
    for p in paths:
        # TextLoader works well for plain Markdown; keeps path in metadata
        loader = TextLoader(str(p), encoding="utf-8")
        docs.extend(loader.load())
    return docs

In [16]:
def main():
    root = Path(MARKDOWN_INPUT)
    paths = load_markdown_paths(root)

    # 1) load raw docs
    docs = load_docs(paths)

    # 2) split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)

    # 3) choose embedding model
    if USE_OPENAI:
        embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


    # 4) build FAISS and persist
    vstore = FAISS.from_documents(chunks, embedding_model)
    Path(EMBEDDINGS_DIRECTORY).mkdir(parents=True, exist_ok=True)
    vstore.save_local(EMBEDDINGS_DIRECTORY)

    print(f"✅ Vector store created with {len(chunks)} chunks and saved to {EMBEDDINGS_DIRECTORY}")

if __name__ == "__main__":
    main()

✅ Vector store created with 104 chunks and saved to ../vstore
